# 05A2. KRX Stock Data Pipeline

## 목적
KRX 공식 Open API를 이용해
한국 주식시장의 개별 종목 일별 데이터를 수집하고 정제한다.

### 수집 대상
- KOSPI
- KOSDAQ
- Open / High / Low / Close
- Volume
- Trading Value
- Market Capitalization

### 전체 흐름
KRX Open API
→ 하루치 응답 검증
→ KOSPI + KOSDAQ 결합
→ 컬럼 / 종목코드 검증
→ Raw 저장
→ 데이터 정제
→ 장기간 수집 방식 설계
→ 종목별 Daily Panel 구축

### 중요
현재 KRX IP 접근 제한 상태이므로
API 호출은 명시적으로 허용하기 전까지 실행하지 않는다.

In [1]:
#05a2-2. 환경 설정
# 목적:프로젝트 경로 설정, krx api Key 로드, raw 데이터 저장 경로 준비
# 중요:이 셀에서는 KRX API 호출 x

from pathlib import Path
import os
import pandas as pd
import requests
from dotenv import load_dotenv

PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)

KRX_STOCK_RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "krx"
    / "stocks"
)

KRX_STOCK_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)


load_dotenv(
    PROJECT_ROOT / ".env"
)

KRX_API_KEY = os.getenv(
    "KRX_API_KEY"
)


print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "KRX API key loaded:",
    bool(KRX_API_KEY)
)

print(
    "Raw directory:",
    KRX_STOCK_RAW_DIR
)

Project root: C:\code\portfolio_optimization
KRX API key loaded: True
Raw directory: C:\code\portfolio_optimization\data\raw\krx\stocks


In [8]:
#05a2-3. API 실행 안전장치
# 현재 KRX IP 제한 상태이므로 False 유지
# 제한 해제 후 하루치 테스트를 할 때만 직접 True로 변경

RUN_KRX_API = True

if RUN_KRX_API:
    print(
        "WARNING: KRX API requests are ENABLED."
    )
else:
    print(
        "SAFE MODE: KRX API requests are DISABLED."
    )

In [3]:
#05a2-4. krx open api endpoint 설정
# KOSPI:유가증권 일별매매정보
# KOSDAQ:코스닥 일별매매정보
# 이 셀 역시 API 요청 x

KRX_BASE_URL = (
    "https://data-dbg.krx.co.kr"
)

KOSPI_DAILY_URL = (
    f"{KRX_BASE_URL}"
    "/svc/apis/sto/stk_bydd_trd"
)

KOSDAQ_DAILY_URL = (
    f"{KRX_BASE_URL}"
    "/svc/apis/sto/ksq_bydd_trd"
)


print(
    "KOSPI endpoint:",
    KOSPI_DAILY_URL
)

print(
    "KOSDAQ endpoint:",
    KOSDAQ_DAILY_URL
)

KOSPI endpoint: https://data-dbg.krx.co.kr/svc/apis/sto/stk_bydd_trd
KOSDAQ endpoint: https://data-dbg.krx.co.kr/svc/apis/sto/ksq_bydd_trd


In [69]:
# 05a2-5. krx 하루치 조회 함수
# timeout 발생 시 최대 3회 재시도

import time


def fetch_krx_daily(
    url,
    date,
    max_retries=3
):

    if not RUN_KRX_API:
        raise RuntimeError(
            "RUN_KRX_API=False"
        )

    if not KRX_API_KEY:
        raise RuntimeError(
            "KRX_API_KEY가 없습니다."
        )

    date_str = pd.Timestamp(
        date
    ).strftime(
        "%Y%m%d"
    )

    headers = {
        "AUTH_KEY": KRX_API_KEY
    }

    params = {
        "basDd": date_str
    }


    for attempt in range(
        1,
        max_retries + 1
    ):

        try:

            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=30
            )

            print(
                f"Date: {date_str} "
                f"HTTP status: {response.status_code}"
            )

            response.raise_for_status()

            data = response.json()

            rows = data.get(
                "OutBlock_1",
                []
            )

            if len(rows) == 0:
                raise RuntimeError(
                    f"No rows returned for {date_str}"
                )

            return pd.DataFrame(
                rows
            )


        except requests.exceptions.ReadTimeout:

            print(
                f"timeout: {date_str} "
                f"({attempt}/{max_retries})"
            )

            if attempt == max_retries:
                raise

            time.sleep(
                3 * attempt
            )

In [5]:
# 05a2-6. 하루치 테스트 날짜 설정
# 기존 연구의 마지막 거래일을 사용
# 현재는 날짜만 정의
# API 요청x

TEST_DATE = "20260915"

print(
    "Test date:",
    TEST_DATE
)

Test date: 20260915


In [9]:
#05a2-7. KOSPI 하루치 테스트
# 제한 해제 후 가장 먼저 실행할 테스트
# 중요:장기간 반복 호출 금지, 우선 딱 하루만 확인

if RUN_KRX_API:

    kospi_raw = fetch_krx_daily(
        url=KOSPI_DAILY_URL,
        date=TEST_DATE
    )

    print(
        "KOSPI shape:",
        kospi_raw.shape
    )

    kospi_raw.head()

else:

    print(
        "SKIPPED: KRX API disabled."
    )

Date: 20260915
HTTP status: 200
KOSPI shape: (943, 15)


In [10]:
#05a2-8. KOSDAQ 하루치 테스트
#KOSPI 테스트 성공 후 동일 날짜의 KOSDAQ 전체 종목 데이터를 확인

if RUN_KRX_API:

    kosdaq_raw = fetch_krx_daily(
        url=KOSDAQ_DAILY_URL,
        date=TEST_DATE
    )

    print(
        "KOSDAQ shape:",
        kosdaq_raw.shape
    )

    kosdaq_raw.head()

else:

    print(
        "SKIPPED: KRX API disabled."
    )

Date: 20260915
HTTP status: 200
KOSDAQ shape: (1820, 15)


In [11]:
# 05a2-9. API 응답 구조 검증
# 목적:반환 컬럼 확인, KOSPI / KOSDAQ Schema 비교

if (
    "kospi_raw" in globals()
    and
    "kosdaq_raw" in globals()
):

    print(
        "KOSPI columns:"
    )

    print(
        kospi_raw.columns.tolist()
    )


    print(
        "\nKOSDAQ columns:"
    )

    print(
        kosdaq_raw.columns.tolist()
    )


    print(
        "\nSame schema:",
        set(kospi_raw.columns)
        ==
        set(kosdaq_raw.columns)
    )

else:

    print(
        "SKIPPED: API data not collected yet."
    )

KOSPI columns:
['BAS_DD', 'ISU_CD', 'ISU_NM', 'MKT_NM', 'SECT_TP_NM', 'TDD_CLSPRC', 'CMPPREVDD_PRC', 'FLUC_RT', 'TDD_OPNPRC', 'TDD_HGPRC', 'TDD_LWPRC', 'ACC_TRDVOL', 'ACC_TRDVAL', 'MKTCAP', 'LIST_SHRS']

KOSDAQ columns:
['BAS_DD', 'ISU_CD', 'ISU_NM', 'MKT_NM', 'SECT_TP_NM', 'TDD_CLSPRC', 'CMPPREVDD_PRC', 'FLUC_RT', 'TDD_OPNPRC', 'TDD_HGPRC', 'TDD_LWPRC', 'ACC_TRDVOL', 'ACC_TRDVAL', 'MKTCAP', 'LIST_SHRS']

Same schema: True


In [12]:
# 05a2-10. KOSPI + KOSDAQ 하루치 데이터 결합
# 목적:한국 주식시장 전체 종목을 하나의 dataframe으로 통합

stock_daily_raw = pd.concat(
    [
        kospi_raw,
        kosdaq_raw
    ],
    ignore_index=True
)

print(
    "KOSPI rows:",
    len(kospi_raw)
)

print(
    "KOSDAQ rows:",
    len(kosdaq_raw)
)

print(
    "Total rows:",
    len(stock_daily_raw)
)

print(
    "Columns:",
    stock_daily_raw.columns.tolist()
)

stock_daily_raw.head()

KOSPI rows: 943
KOSDAQ rows: 1820
Total rows: 2763
Columns: ['BAS_DD', 'ISU_CD', 'ISU_NM', 'MKT_NM', 'SECT_TP_NM', 'TDD_CLSPRC', 'CMPPREVDD_PRC', 'FLUC_RT', 'TDD_OPNPRC', 'TDD_HGPRC', 'TDD_LWPRC', 'ACC_TRDVOL', 'ACC_TRDVAL', 'MKTCAP', 'LIST_SHRS']


,BAS_DD,ISU_CD,ISU_NM,MKT_NM,SECT_TP_NM,TDD_CLSPRC,CMPPREVDD_PRC,FLUC_RT,TDD_OPNPRC,TDD_HGPRC,TDD_LWPRC,ACC_TRDVOL,ACC_TRDVAL,MKTCAP,LIST_SHRS
0,20260915,095570,AJ네트웍스,KOSPI,,4155,-40,-0.95,4210,4210,4140,51907,215709277,188025213645,45252759
1,20260915,006840,AK홀딩스,KOSPI,,7480,-50,-0.66,7530,7830,7330,5873,43716810,99091756280,13247561
2,20260915,027410,BGF,KOSPI,,3745,-20,-0.53,3760,3785,3720,106101,397535702,358459382295,95716791
3,20260915,282330,BGF리테일,KOSPI,,141800,-4100,-2.81,146600,146600,140100,31249,4429016500,2450857870800,17283906
4,20260915,138930,BNK금융지주,KOSPI,,16040,30,0.19,15910,16160,15790,856790,13754863115,4921709606040,306839751


In [13]:
# 05a2-11. 종목코드 형식 확인
# 목적:ISU_CD가 기존 Membership ticker와 연결 가능한지 확인

stock_daily_raw[
    [
        "ISU_CD",
        "ISU_NM",
        "MKT_NM"
    ]
].head(20)

,ISU_CD,ISU_NM,MKT_NM
0,095570,AJ네트웍스,KOSPI
1,006840,AK홀딩스,KOSPI
2,027410,BGF,KOSPI
3,282330,BGF리테일,KOSPI
4,138930,BNK금융지주,KOSPI
5,001460,BYC,KOSPI
6,001465,BYC우,KOSPI
7,001040,CJ,KOSPI
8,079160,CJ CGV,KOSPI
9,00104K,CJ4우(전환),KOSPI


In [14]:
stock_daily_raw[
    stock_daily_raw[
        "ISU_NM"
    ].str.contains(
        "삼성전자",
        na=False
    )
][
    [
        "ISU_CD",
        "ISU_NM",
        "MKT_NM"
    ]
]

,ISU_CD,ISU_NM,MKT_NM
457,005930,삼성전자,KOSPI
458,005935,삼성전자우,KOSPI


In [15]:
# 05a2-12. 하루치 Raw 데이터 저장
# 목적:API 응답 원본 보존, 이후 API 재호출 없이 반복 분석

TEST_RAW_PATH = (
    KRX_STOCK_RAW_DIR
    / f"krx_stock_daily_{TEST_DATE}.csv"
)

stock_daily_raw.to_csv(
    TEST_RAW_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    TEST_RAW_PATH
)

Saved: C:\code\portfolio_optimization\data\raw\krx\stocks\krx_stock_daily_20260915.csv


In [16]:
# 05A2-13. 데이터 타입 확인
# 목적:숫자형 변환 전 원본 타입 점검

stock_daily_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2763 entries, 0 to 2762
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   BAS_DD         2763 non-null   object
 1   ISU_CD         2763 non-null   object
 2   ISU_NM         2763 non-null   object
 3   MKT_NM         2763 non-null   object
 4   SECT_TP_NM     2763 non-null   object
 5   TDD_CLSPRC     2763 non-null   object
 6   CMPPREVDD_PRC  2763 non-null   object
 7   FLUC_RT        2763 non-null   object
 8   TDD_OPNPRC     2763 non-null   object
 9   TDD_HGPRC      2763 non-null   object
 10  TDD_LWPRC      2763 non-null   object
 11  ACC_TRDVOL     2763 non-null   object
 12  ACC_TRDVAL     2763 non-null   object
 13  MKTCAP         2763 non-null   object
 14  LIST_SHRS      2763 non-null   object
dtypes: object(15)
memory usage: 323.9+ KB


In [17]:
stock_daily_raw[
    [
        "TDD_CLSPRC",
        "TDD_OPNPRC",
        "TDD_HGPRC",
        "TDD_LWPRC",
        "ACC_TRDVOL",
        "ACC_TRDVAL",
        "MKTCAP",
        "LIST_SHRS"
    ]
].head()

,TDD_CLSPRC,TDD_OPNPRC,TDD_HGPRC,TDD_LWPRC,ACC_TRDVOL,ACC_TRDVAL,MKTCAP,LIST_SHRS
0,4155,4210,4210,4140,51907,215709277,188025213645,45252759
1,7480,7530,7830,7330,5873,43716810,99091756280,13247561
2,3745,3760,3785,3720,106101,397535702,358459382295,95716791
3,141800,146600,146600,140100,31249,4429016500,2450857870800,17283906
4,16040,15910,16160,15790,856790,13754863115,4921709606040,306839751


In [18]:
# 05a2-14. 컬럼명 표준화
# 목적:KRX 원본 컬럼명을 프로젝트 공통 이름으로 변경, 
# raw 데이터는 그대로 보존하고 별도 dataframe에서 정제
# 중요:ticker는 문자열 유지

stock_daily_clean = (
    stock_daily_raw
    .rename(
        columns={
            "BAS_DD": "date",
            "ISU_CD": "ticker",
            "ISU_NM": "name",
            "MKT_NM": "market",
            "SECT_TP_NM": "security_type",
            "TDD_CLSPRC": "close",
            "CMPPREVDD_PRC": "change",
            "FLUC_RT": "change_pct",
            "TDD_OPNPRC": "open",
            "TDD_HGPRC": "high",
            "TDD_LWPRC": "low",
            "ACC_TRDVOL": "volume",
            "ACC_TRDVAL": "trading_value",
            "MKTCAP": "market_cap",
            "LIST_SHRS": "listed_shares"
        }
    )
    .copy()
)

print(
    stock_daily_clean.columns.tolist()
)

stock_daily_clean.head()

['date', 'ticker', 'name', 'market', 'security_type', 'close', 'change', 'change_pct', 'open', 'high', 'low', 'volume', 'trading_value', 'market_cap', 'listed_shares']


,date,ticker,name,market,security_type,close,change,change_pct,open,high,low,volume,trading_value,market_cap,listed_shares
0,20260915,095570,AJ네트웍스,KOSPI,,4155,-40,-0.95,4210,4210,4140,51907,215709277,188025213645,45252759
1,20260915,006840,AK홀딩스,KOSPI,,7480,-50,-0.66,7530,7830,7330,5873,43716810,99091756280,13247561
2,20260915,027410,BGF,KOSPI,,3745,-20,-0.53,3760,3785,3720,106101,397535702,358459382295,95716791
3,20260915,282330,BGF리테일,KOSPI,,141800,-4100,-2.81,146600,146600,140100,31249,4429016500,2450857870800,17283906
4,20260915,138930,BNK금융지주,KOSPI,,16040,30,0.19,15910,16160,15790,856790,13754863115,4921709606040,306839751


In [19]:
# 05a2-15. 데이터 타입 변환
# date   -> datetime
# ticker -> string 유지
# 가격/거래량/시총 -> numeric

stock_daily_clean["date"] = pd.to_datetime(
    stock_daily_clean["date"],
    format="%Y%m%d"
)

stock_daily_clean["ticker"] = (
    stock_daily_clean["ticker"]
    .astype(str)
    .str.strip()
)


NUMERIC_COLUMNS = [
    "close",
    "change",
    "change_pct",
    "open",
    "high",
    "low",
    "volume",
    "trading_value",
    "market_cap",
    "listed_shares"
]


for col in NUMERIC_COLUMNS:

    stock_daily_clean[col] = pd.to_numeric(
        stock_daily_clean[col],
        errors="coerce"
    )


stock_daily_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2763 entries, 0 to 2762
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           2763 non-null   datetime64[ns]
 1   ticker         2763 non-null   object        
 2   name           2763 non-null   object        
 3   market         2763 non-null   object        
 4   security_type  2763 non-null   object        
 5   close          2763 non-null   int64         
 6   change         2763 non-null   int64         
 7   change_pct     2763 non-null   float64       
 8   open           2763 non-null   int64         
 9   high           2763 non-null   int64         
 10  low            2763 non-null   int64         
 11  volume         2763 non-null   int64         
 12  trading_value  2763 non-null   int64         
 13  market_cap     2763 non-null   int64         
 14  listed_shares  2763 non-null   int64         
dtypes: datetime64[ns](1),

In [20]:
# 05a2-16. 결측치 및 중복 검사
# 목적:ticker 중복 여부 확인, 핵심 숫자 데이터 결측 확인, 시장별 데이터 개수 확인

print(
    "Rows:",
    len(stock_daily_clean)
)

print(
    "\nMissing values:"
)

print(
    stock_daily_clean
    .isna()
    .sum()
)


duplicate_tickers = (
    stock_daily_clean[
        ["date", "ticker"]
    ]
    .duplicated()
    .sum()
)

print(
    "\nDuplicate date+ticker:",
    duplicate_tickers
)


print(
    "\nRows by market:"
)

print(
    stock_daily_clean[
        "market"
    ]
    .value_counts()
)

Rows: 2763

Missing values:
date             0
ticker           0
name             0
market           0
security_type    0
close            0
change           0
change_pct       0
open             0
high             0
low              0
volume           0
trading_value    0
market_cap       0
listed_shares    0
dtype: int64

Duplicate date+ticker: 0

Rows by market:
market
KOSDAQ    1820
KOSPI      943
Name: count, dtype: int64


In [21]:
# 05a2-17. OHLC 가격 논리 검증
# 목적:실제 거래된 종목에서 가격 관계가 논리적인지 확인
# 정상 관계:
# high >= open
# high >= close
# low  <= open
# low  <= close
# 거래량 0인 종목은 별도 관찰

traded = (
    stock_daily_clean[
        stock_daily_clean["volume"] > 0
    ]
    .copy()
)


invalid_ohlc = traded[
    (
        traded["high"] < traded["open"]
    )
    |
    (
        traded["high"] < traded["close"]
    )
    |
    (
        traded["low"] > traded["open"]
    )
    |
    (
        traded["low"] > traded["close"]
    )
]


print(
    "Actually traded rows:",
    len(traded)
)

print(
    "Invalid OHLC rows:",
    len(invalid_ohlc)
)

invalid_ohlc[
    [
        "ticker",
        "name",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
].head(20)

Actually traded rows: 2648
Invalid OHLC rows: 0


,ticker,name,open,high,low,close,volume


In [22]:
# 05a2-18. 무거래 종목 확인
# 목적:volume = 0 인 종목 수 확인, 거래정지 등 특수 케이스를 향후 feature 처리 시 구분

no_trade = (
    stock_daily_clean[
        stock_daily_clean["volume"] == 0
    ]
    .copy()
)

print(
    "No-trade rows:",
    len(no_trade)
)

no_trade[
    [
        "ticker",
        "name",
        "market",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "trading_value"
    ]
].head(20)

No-trade rows: 115


,ticker,name,market,open,high,low,close,volume,trading_value
22,000300,DH오토넥스,KOSPI,0,0,0,4200,0,0
83,009440,KC그린홀딩스,KOSPI,0,0,0,756,0,0
84,119650,KC코트렐,KOSPI,0,0,0,918,0,0
184,011810,STX,KOSPI,0,0,0,3530,0,0
216,017040,광명전기,KOSPI,0,0,0,932,0,0
228,001570,금양,KOSPI,0,0,0,9900,0,0
267,145210,다이나믹디자인,KOSPI,0,0,0,2240,0,0
315,069460,대호에이엘,KOSPI,0,0,0,2660,0,0
420,002410,범양건영,KOSPI,0,0,0,1935,0,0
430,005030,부산주공,KOSPI,0,0,0,486,0,0


In [23]:
# 05a2-19. KRX300 Membership ticker 연결 검증
# 목적: 기존 KRX300 Membership ticker, KRX Open API ISU_CD
# 두 데이터가 동일한 ticker key로 연결되는지 확인
# API 추가 호출 없음

MEMBERSHIP_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "krx"
    / "universe"
    / "krx300_monthly_membership_snapshots.csv"
)


monthly_membership = pd.read_csv(
    MEMBERSHIP_PATH,
    dtype={
        "ticker": str,
        "index_code": str
    }
)

monthly_membership["date"] = pd.to_datetime(
    monthly_membership["date"]
)

monthly_membership["ticker"] = (
    monthly_membership["ticker"]
    .astype(str)
    .str.strip()
)

In [24]:
#05a2-20. 테스트 날짜 membership 추출

test_date_timestamp = pd.Timestamp(
    TEST_DATE
)

test_membership = (
    monthly_membership[
        monthly_membership["date"]
        == test_date_timestamp
    ]
    .copy()
)

print(
    "KRX300 membership rows:",
    len(test_membership)
)

test_membership.head()

KRX300 membership rows: 301


,date,ticker,index_code,index_name,member_count
30931,2026-09-15,005930,5300,KRX 300,301
30932,2026-09-15,000660,5300,KRX 300,301
30933,2026-09-15,402340,5300,KRX 300,301
30934,2026-09-15,009150,5300,KRX 300,301
30935,2026-09-15,373220,5300,KRX 300,301


In [25]:
# 05a2-21. membership ↔ stock market data 연결 검증
# 목적: KRX300 구성종목이 open api 시장 데이터에 존재하는지 확인

membership_match = (
    test_membership[
        [
            "date",
            "ticker"
        ]
    ]
    .merge(
        stock_daily_clean[
            [
                "date",
                "ticker",
                "name",
                "market",
                "close",
                "volume",
                "trading_value",
                "market_cap"
            ]
        ],
        on=[
            "date",
            "ticker"
        ],
        how="left",
        indicator=True
    )
)


print(
    membership_match[
        "_merge"
    ]
    .value_counts()
)


match_rate = (
    membership_match[
        "_merge"
    ]
    .eq("both")
    .mean()
)


print(
    "\nMatch rate:",
    f"{match_rate:.2%}"
)

_merge
both          301
left_only       0
right_only      0
Name: count, dtype: int64

Match rate: 100.00%


In [26]:
# 05a2-22. 하루치 clean 데이터 저장
# Raw: KRX API 원본 그대로
# Clean:표준 컬럼명 + 타입 변환 완료

KRX_STOCK_CLEAN_DIR = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "krx"
    / "stocks"
)

KRX_STOCK_CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TEST_CLEAN_PATH = (
    KRX_STOCK_CLEAN_DIR
    / f"krx_stock_daily_{TEST_DATE}_clean.csv"
)


stock_daily_clean.to_csv(
    TEST_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    TEST_CLEAN_PATH
)

Saved: C:\code\portfolio_optimization\data\clean\krx\stocks\krx_stock_daily_20260915_clean.csv


In [27]:
# 05a2-23. 장기간 수집 대상 거래일 준비
# 기존 krx300 거래일만 사용

KRX300_INDEX_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "krx"
    / "index"
    / "krx300_index_daily_pykrx.csv"
)

krx300_calendar = pd.read_csv(
    KRX300_INDEX_PATH
)

krx300_calendar["날짜"] = pd.to_datetime(
    krx300_calendar["날짜"]
)

collection_dates = (
    krx300_calendar["날짜"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print(
    "collection dates:",
    len(collection_dates)
)

print(
    "start:",
    collection_dates.min()
)

print(
    "end:",
    collection_dates.max()
)

collection dates: 2112
start: 2018-02-05 00:00:00
end: 2026-09-15 00:00:00


In [28]:
# 05a2-24. 이미 저장된 날짜 확인
# 같은 날짜를 api로 다시 요청하지 않음

existing_files = list(
    KRX_STOCK_RAW_DIR.glob(
        "krx_stock_daily_*.csv"
    )
)

existing_dates = set()

for path in existing_files:

    date_text = (
        path.stem
        .replace(
            "krx_stock_daily_",
            ""
        )
    )

    if (
        len(date_text) == 8
        and
        date_text.isdigit()
    ):
        existing_dates.add(
            pd.Timestamp(date_text)
        )


print(
    "existing raw dates:",
    len(existing_dates)
)

print(
    sorted(existing_dates)[-5:]
    if existing_dates
    else []
)

existing raw dates: 1
[Timestamp('2026-09-15 00:00:00')]


In [29]:
# 05a2-25. 미수집 날짜 계산

pending_dates = [
    date
    for date in collection_dates
    if date not in existing_dates
]

print(
    "total dates:",
    len(collection_dates)
)

print(
    "already collected:",
    len(existing_dates)
)

print(
    "pending:",
    len(pending_dates)
)

print(
    "first pending:",
    pending_dates[:5]
)

total dates: 2112
already collected: 1
pending: 2111
first pending: [Timestamp('2018-02-05 00:00:00'), Timestamp('2018-02-06 00:00:00'), Timestamp('2018-02-07 00:00:00'), Timestamp('2018-02-08 00:00:00'), Timestamp('2018-02-09 00:00:00')]


In [30]:
# 05a2-26. 하루치 전체 주식 데이터 수집

def fetch_krx_stock_day(date):

    kospi = fetch_krx_daily(
        url=KOSPI_DAILY_URL,
        date=date
    )

    kosdaq = fetch_krx_daily(
        url=KOSDAQ_DAILY_URL,
        date=date
    )

    combined = pd.concat(
        [
            kospi,
            kosdaq
        ],
        ignore_index=True
    )

    return combined

In [31]:
# 05a2-27. 하루치 raw 저장 함수

def save_krx_stock_day_raw(
    df,
    date
):

    date_str = pd.Timestamp(
        date
    ).strftime(
        "%Y%m%d"
    )

    save_path = (
        KRX_STOCK_RAW_DIR
        / f"krx_stock_daily_{date_str}.csv"
    )

    df.to_csv(
        save_path,
        index=False,
        encoding="utf-8-sig"
    )

    return save_path

In [32]:
# 05a2-28. 소규모 수집 테스트 설정
# 전체 수집 전 3일만 확인

RUN_BATCH_TEST = False

batch_test_dates = (
    pending_dates[:3]
)

print(
    batch_test_dates
)

[Timestamp('2018-02-05 00:00:00'), Timestamp('2018-02-06 00:00:00'), Timestamp('2018-02-07 00:00:00')]


In [35]:
# 05a2-29. 3일 batch 테스트
# 실패 시 즉시 중단, 성공한 날짜는 raw 저장

RUN_BATCH_TEST = True

batch_results = []


if RUN_BATCH_TEST:

    for date in batch_test_dates:

        date_str = pd.Timestamp(
            date
        ).strftime(
            "%Y%m%d"
        )

        save_path = (
            KRX_STOCK_RAW_DIR
            / f"krx_stock_daily_{date_str}.csv"
        )

        if save_path.exists():

            print(
                f"skip: {date_str}"
            )

            batch_results.append({
                "date": date_str,
                "status": "skipped"
            })

            continue


        print(
            f"collect: {date_str}"
        )

        try:

            daily_raw = fetch_krx_stock_day(
                date
            )

            save_krx_stock_day_raw(
                daily_raw,
                date
            )

            print(
                f"saved: {date_str}, rows: {len(daily_raw)}"
            )

            batch_results.append({
                "date": date_str,
                "status": "saved",
                "rows": len(daily_raw)
            })


        except Exception as e:

            print(
                f"error: {date_str}"
            )

            print(
                type(e).__name__,
                str(e)
            )

            batch_results.append({
                "date": date_str,
                "status": "error"
            })

            break


batch_results_df = pd.DataFrame(
    batch_results
)

batch_results_df

skip: 20180205
skip: 20180206
skip: 20180207


,date,status
0,20180205,skipped
1,20180206,skipped
2,20180207,skipped


In [34]:
# 05a2-30. batch 저장 결과 확인

saved_test_files = sorted(
    KRX_STOCK_RAW_DIR.glob(
        "krx_stock_daily_2018020*.csv"
    )
)

for path in saved_test_files:
    print(
        path.name
    )

krx_stock_daily_20180205.csv
krx_stock_daily_20180206.csv
krx_stock_daily_20180207.csv


In [37]:
# 05a2-32. 3일 raw 데이터 기본 검증

batch_validation = []

for date in batch_test_dates:

    date_str = pd.Timestamp(
        date
    ).strftime(
        "%Y%m%d"
    )

    path = (
        KRX_STOCK_RAW_DIR
        / f"krx_stock_daily_{date_str}.csv"
    )

    if not path.exists():
        continue

    df = pd.read_csv(
        path,
        dtype={
            "ISU_CD": str
        }
    )

    batch_validation.append({
        "date": date_str,
        "rows": len(df),
        "duplicate_ticker": df["ISU_CD"].duplicated().sum(),
        "missing_ticker": df["ISU_CD"].isna().sum()
    })


batch_validation_df = pd.DataFrame(
    batch_validation
)

batch_validation_df

,date,rows,duplicate_ticker,missing_ticker
0,20180205,2162,0,0
1,20180206,2162,0,0
2,20180207,2161,0,0


In [38]:
# 05a2-33. 20일 batch 테스트

batch_20_dates = [
    date
    for date in collection_dates
    if date not in existing_dates
][:20]

print(
    "batch size:",
    len(batch_20_dates)
)

print(
    "start:",
    batch_20_dates[0]
)

print(
    "end:",
    batch_20_dates[-1]
)

batch size: 20
start: 2018-02-05 00:00:00
end: 2018-03-07 00:00:00


In [39]:
# 05a2-34. 저장된 날짜 목록 갱신

existing_files = list(
    KRX_STOCK_RAW_DIR.glob(
        "krx_stock_daily_*.csv"
    )
)

existing_dates = set()

for path in existing_files:

    date_text = (
        path.stem
        .replace(
            "krx_stock_daily_",
            ""
        )
    )

    if (
        len(date_text) == 8
        and
        date_text.isdigit()
    ):
        existing_dates.add(
            pd.Timestamp(date_text)
        )


pending_dates = [
    date
    for date in collection_dates
    if date not in existing_dates
]


print(
    "collected:",
    len(existing_dates)
)

print(
    "pending:",
    len(pending_dates)
)

print(
    "next:",
    pending_dates[:5]
)

collected: 4
pending: 2108
next: [Timestamp('2018-02-08 00:00:00'), Timestamp('2018-02-09 00:00:00'), Timestamp('2018-02-12 00:00:00'), Timestamp('2018-02-13 00:00:00'), Timestamp('2018-02-14 00:00:00')]


In [40]:
# 05a2-35. 20일 batch 수집

batch_20_dates = pending_dates[:20]

batch_20_results = []


for date in batch_20_dates:

    date_str = pd.Timestamp(
        date
    ).strftime(
        "%Y%m%d"
    )

    save_path = (
        KRX_STOCK_RAW_DIR
        / f"krx_stock_daily_{date_str}.csv"
    )

    if save_path.exists():

        print(
            f"skip: {date_str}"
        )

        continue


    print(
        f"collect: {date_str}"
    )

    try:

        daily_raw = fetch_krx_stock_day(
            date
        )

        save_krx_stock_day_raw(
            daily_raw,
            date
        )

        print(
            f"saved: {date_str}, rows: {len(daily_raw)}"
        )

        batch_20_results.append({
            "date": date_str,
            "status": "saved",
            "rows": len(daily_raw)
        })


    except Exception as e:

        print(
            f"error: {date_str}"
        )

        print(
            type(e).__name__,
            str(e)
        )

        batch_20_results.append({
            "date": date_str,
            "status": "error"
        })

        break


batch_20_results_df = pd.DataFrame(
    batch_20_results
)

batch_20_results_df

collect: 20180208
Date: 20180208
HTTP status: 200
Date: 20180208
HTTP status: 200
saved: 20180208, rows: 2162
collect: 20180209
Date: 20180209
HTTP status: 200
Date: 20180209
HTTP status: 200
saved: 20180209, rows: 2162
collect: 20180212
Date: 20180212
HTTP status: 200
Date: 20180212
HTTP status: 200
saved: 20180212, rows: 2164
collect: 20180213
Date: 20180213
HTTP status: 200
Date: 20180213
HTTP status: 200
saved: 20180213, rows: 2165
collect: 20180214
Date: 20180214
HTTP status: 200
Date: 20180214
HTTP status: 200
saved: 20180214, rows: 2164
collect: 20180219
Date: 20180219
HTTP status: 200
Date: 20180219
HTTP status: 200
saved: 20180219, rows: 2164
collect: 20180220
Date: 20180220
HTTP status: 200
Date: 20180220
HTTP status: 200
saved: 20180220, rows: 2164
collect: 20180221
Date: 20180221
HTTP status: 200
Date: 20180221
HTTP status: 200
saved: 20180221, rows: 2165
collect: 20180222
Date: 20180222
HTTP status: 200
Date: 20180222
HTTP status: 200
saved: 20180222, rows: 2166
collect: 2

,date,status,rows
0,20180208,saved,2162
1,20180209,saved,2162
2,20180212,saved,2164
3,20180213,saved,2165
4,20180214,saved,2164
5,20180219,saved,2164
6,20180220,saved,2164
7,20180221,saved,2165
8,20180222,saved,2166
9,20180223,saved,2165


In [41]:
# 05a2-36. 20일 batch 결과 검증

print(
    batch_20_results_df[
        "status"
    ].value_counts()
)

print(
    "\nrows:"
)

print(
    batch_20_results_df[
        "rows"
    ].describe()
)

status
saved    20
Name: count, dtype: int64

rows:
count      20.000000
mean     2164.100000
std         1.020836
min      2162.000000
25%      2164.000000
50%      2164.000000
75%      2165.000000
max      2166.000000
Name: rows, dtype: float64


In [58]:
# 05a2-37. 전체 수집 설정
# 100거래일씩 수집하고 다음 실행에서 이어서 진행

import time

MAX_DATES_PER_RUN = 500

REQUEST_DELAY_SECONDS = 0.3

In [74]:
# 05a2-38. 현재 수집 상태 갱신

existing_files = list(
    KRX_STOCK_RAW_DIR.glob(
        "krx_stock_daily_*.csv"
    )
)

existing_dates = set()

for path in existing_files:

    date_text = (
        path.stem
        .replace(
            "krx_stock_daily_",
            ""
        )
    )

    if (
        len(date_text) == 8
        and
        date_text.isdigit()
    ):
        existing_dates.add(
            pd.Timestamp(date_text)
        )


pending_dates = [
    date
    for date in collection_dates
    if date not in existing_dates
]


print(
    "total:",
    len(collection_dates)
)

print(
    "collected:",
    len(existing_dates)
)

print(
    "pending:",
    len(pending_dates)
)

print(
    "progress:",
    f"{len(existing_dates) / len(collection_dates):.2%}"
)

total: 2112
collected: 1767
pending: 345
progress: 83.66%


In [60]:
# 05a2-39. 수집 로그 설정

COLLECTION_LOG_PATH = (
    KRX_STOCK_RAW_DIR
    / "collection_log.csv"
)


if COLLECTION_LOG_PATH.exists():

    collection_log = pd.read_csv(
        COLLECTION_LOG_PATH,
        dtype={
            "date": str
        }
    )

else:

    collection_log = pd.DataFrame(
        columns=[
            "date",
            "status",
            "rows"
        ]
    )


collection_log.tail()

,date,status,rows
395,20191023,saved,2282
396,20191024,saved,2283
397,20191025,saved,2284
398,20191028,saved,2284
399,20191029,saved,2286


In [75]:
# 05a2-40. 500일 batch 수집
# 저장된 날짜는 skip, 오류 발생 시 즉시 중단

run_dates = (
    pending_dates[
        :MAX_DATES_PER_RUN
    ]
)

run_results = []


print(
    "dates this run:",
    len(run_dates)
)


for i, date in enumerate(
    run_dates,
    start=1
):

    date_str = pd.Timestamp(
        date
    ).strftime(
        "%Y%m%d"
    )

    save_path = (
        KRX_STOCK_RAW_DIR
        / f"krx_stock_daily_{date_str}.csv"
    )


    if save_path.exists():

        print(
            f"[{i}/{len(run_dates)}] skip: {date_str}"
        )

        continue


    print(
        f"[{i}/{len(run_dates)}] collect: {date_str}"
    )


    try:

        daily_raw = fetch_krx_stock_day(
            date
        )


        if daily_raw.empty:

            raise RuntimeError(
                "empty dataframe"
            )


        duplicate_ticker = (
            daily_raw[
                "ISU_CD"
            ]
            .duplicated()
            .sum()
        )


        if duplicate_ticker > 0:

            raise RuntimeError(
                f"duplicate ticker: {duplicate_ticker}"
            )


        missing_ticker = (
            daily_raw[
                "ISU_CD"
            ]
            .isna()
            .sum()
        )


        if missing_ticker > 0:

            raise RuntimeError(
                f"missing ticker: {missing_ticker}"
            )


        save_krx_stock_day_raw(
            daily_raw,
            date
        )


        print(
            f"saved: {date_str}, rows: {len(daily_raw)}"
        )


        run_results.append({
            "date": date_str,
            "status": "saved",
            "rows": len(daily_raw)
        })


        time.sleep(
            REQUEST_DELAY_SECONDS
        )


    except Exception as e:

        print(
            f"error: {date_str}"
        )

        print(
            type(e).__name__,
            str(e)
        )


        run_results.append({
            "date": date_str,
            "status": "error",
            "rows": None
        })


        break

dates this run: 345
[1/345] collect: 20250416
Date: 20250416 HTTP status: 200
Date: 20250416 HTTP status: 200
saved: 20250416, rows: 2757
[2/345] collect: 20250417
Date: 20250417 HTTP status: 200
Date: 20250417 HTTP status: 200
saved: 20250417, rows: 2757
[3/345] collect: 20250418
Date: 20250418 HTTP status: 200
Date: 20250418 HTTP status: 200
saved: 20250418, rows: 2757
[4/345] collect: 20250421
Date: 20250421 HTTP status: 200
Date: 20250421 HTTP status: 200
saved: 20250421, rows: 2757
[5/345] collect: 20250422
Date: 20250422 HTTP status: 200
Date: 20250422 HTTP status: 200
saved: 20250422, rows: 2757
[6/345] collect: 20250423
Date: 20250423 HTTP status: 200
Date: 20250423 HTTP status: 200
saved: 20250423, rows: 2757
[7/345] collect: 20250424
Date: 20250424 HTTP status: 200
Date: 20250424 HTTP status: 200
saved: 20250424, rows: 2757
[8/345] collect: 20250425
Date: 20250425 HTTP status: 200
Date: 20250425 HTTP status: 200
saved: 20250425, rows: 2757
[9/345] collect: 20250428
Date: 2025

In [76]:
# 05a2-41. 수집 로그 저장

run_results_df = pd.DataFrame(
    run_results
)


collection_log = pd.concat(
    [
        collection_log,
        run_results_df
    ],
    ignore_index=True
)

collection_log = (
    collection_log
    .drop_duplicates(
        subset=[
            "date"
        ],
        keep="last"
    )
    .sort_values(
        "date"
    )
    .reset_index(
        drop=True
    )
)


collection_log.to_csv(
    COLLECTION_LOG_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    collection_log[
        "status"
    ]
    .value_counts()
)

collection_log.tail()

status
saved    2088
Name: count, dtype: int64


,date,status,rows
2083,20260908,saved,2765.0
2084,20260909,saved,2765.0
2085,20260910,saved,2766.0
2086,20260911,saved,2765.0
2087,20260914,saved,2764.0


In [77]:
# 05a2-42. 수집 진행률 확인

saved_dates = set()

for path in KRX_STOCK_RAW_DIR.glob(
    "krx_stock_daily_*.csv"
):

    date_text = (
        path.stem
        .replace(
            "krx_stock_daily_",
            ""
        )
    )

    if (
        len(date_text) == 8
        and
        date_text.isdigit()
    ):

        saved_dates.add(
            pd.Timestamp(
                date_text
            )
        )


remaining_dates = [
    date
    for date in collection_dates
    if date not in saved_dates
]


print(
    "total:",
    len(collection_dates)
)

print(
    "saved:",
    len(saved_dates)
)

print(
    "remaining:",
    len(remaining_dates)
)

print(
    "progress:",
    f"{len(saved_dates) / len(collection_dates):.2%}"
)

total: 2112
saved: 2112
remaining: 0
progress: 100.00%


In [78]:
# 05a2-43. 전체 raw 파일 목록 확인

raw_files = sorted(
    KRX_STOCK_RAW_DIR.glob(
        "krx_stock_daily_*.csv"
    )
)

print(
    "raw files:",
    len(raw_files)
)

print(
    "first:",
    raw_files[0].name
)

print(
    "last:",
    raw_files[-1].name
)

raw files: 2112
first: krx_stock_daily_20180205.csv
last: krx_stock_daily_20260915.csv


In [79]:
# 05a2-44. 전체 raw 파일 통합

all_daily_data = []


for path in raw_files:

    df = pd.read_csv(
        path,
        dtype={
            "ISU_CD": str
        }
    )

    all_daily_data.append(
        df
    )


stock_panel_raw = pd.concat(
    all_daily_data,
    ignore_index=True
)


print(
    "shape:",
    stock_panel_raw.shape
)

print(
    "dates:",
    stock_panel_raw["BAS_DD"].nunique()
)

print(
    "tickers:",
    stock_panel_raw["ISU_CD"].nunique()
)

stock_panel_raw.head()

shape: (5278578, 15)
dates: 2112
tickers: 3186


,BAS_DD,ISU_CD,ISU_NM,MKT_NM,SECT_TP_NM,TDD_CLSPRC,CMPPREVDD_PRC,FLUC_RT,TDD_OPNPRC,TDD_HGPRC,TDD_LWPRC,ACC_TRDVOL,ACC_TRDVAL,MKTCAP,LIST_SHRS
0,20180205,095570,AJ네트웍스,KOSPI,NaN,7800,-10,-0.13,7700,7860,7620,83893,652547630,365213901000,46822295
1,20180205,068400,AJ렌터카,KOSPI,NaN,11300,-500,-4.24,11450,11800,11200,416965,4793034700,250253190000,22146300
2,20180205,006840,AK홀딩스,KOSPI,NaN,81900,-1800,-2.15,83700,83700,81500,18640,1535892800,1084975245900,13247561
3,20180205,027410,BGF,KOSPI,NaN,14950,-350,-2.29,14900,15200,14850,363936,5462105850,482342599050,32263719
4,20180205,282330,BGF리테일,KOSPI,NaN,212000,-2000,-0.93,218000,221000,210000,24307,5169440500,3664188072000,17283906


In [80]:
# 05a2-45. 전체 stock panel 컬럼명 표준화

stock_panel_clean = stock_panel_raw.rename(
    columns={
        "BAS_DD": "date",
        "ISU_CD": "ticker",
        "ISU_NM": "name",
        "MKT_NM": "market",
        "SECT_TP_NM": "security_type",
        "TDD_CLSPRC": "close",
        "CMPPREVDD_PRC": "change",
        "FLUC_RT": "change_pct",
        "TDD_OPNPRC": "open",
        "TDD_HGPRC": "high",
        "TDD_LWPRC": "low",
        "ACC_TRDVOL": "volume",
        "ACC_TRDVAL": "trading_value",
        "MKTCAP": "market_cap",
        "LIST_SHRS": "listed_shares"
    }
)

print(
    stock_panel_clean.columns.tolist()
)

['date', 'ticker', 'name', 'market', 'security_type', 'close', 'change', 'change_pct', 'open', 'high', 'low', 'volume', 'trading_value', 'market_cap', 'listed_shares']


In [81]:
# 05a2-46. 전체 stock panel 타입 변환

stock_panel_clean["date"] = pd.to_datetime(
    stock_panel_clean["date"].astype(str),
    format="%Y%m%d"
)

stock_panel_clean["ticker"] = (
    stock_panel_clean["ticker"]
    .astype(str)
    .str.strip()
)


NUMERIC_COLUMNS = [
    "close",
    "change",
    "change_pct",
    "open",
    "high",
    "low",
    "volume",
    "trading_value",
    "market_cap",
    "listed_shares"
]


for col in NUMERIC_COLUMNS:

    stock_panel_clean[col] = pd.to_numeric(
        stock_panel_clean[col],
        errors="coerce"
    )


stock_panel_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5278578 entries, 0 to 5278577
Data columns (total 15 columns):
 #   Column         Dtype         
---  ------         -----         
 0   date           datetime64[ns]
 1   ticker         object        
 2   name           object        
 3   market         object        
 4   security_type  object        
 5   close          int64         
 6   change         int64         
 7   change_pct     float64       
 8   open           int64         
 9   high           int64         
 10  low            int64         
 11  volume         int64         
 12  trading_value  int64         
 13  market_cap     int64         
 14  listed_shares  int64         
dtypes: datetime64[ns](1), float64(1), int64(9), object(4)
memory usage: 604.1+ MB


In [82]:
# 05a2-47. 전체 panel 기본 검증

print(
    "rows:",
    len(stock_panel_clean)
)

print(
    "dates:",
    stock_panel_clean["date"].nunique()
)

print(
    "tickers:",
    stock_panel_clean["ticker"].nunique()
)

print(
    "start:",
    stock_panel_clean["date"].min()
)

print(
    "end:",
    stock_panel_clean["date"].max()
)

rows: 5278578
dates: 2112
tickers: 3186
start: 2018-02-05 00:00:00
end: 2026-09-15 00:00:00


In [83]:
# 05a2-48. 날짜별 종목 수 확인

rows_by_date = (
    stock_panel_clean
    .groupby("date")
    .size()
)

print(
    rows_by_date.describe()
)

print(
    "\nmin date:"
)

print(
    rows_by_date.nsmallest(10)
)

print(
    "\nmax date:"
)

print(
    rows_by_date.nlargest(10)
)

count    2112.000000
mean     2499.326705
std       202.485602
min      2161.000000
25%      2325.000000
50%      2498.000000
75%      2696.250000
max      2789.000000
dtype: float64

min date:
date
2018-02-07    2161
2018-02-05    2162
2018-02-06    2162
2018-02-08    2162
2018-02-09    2162
2018-03-13    2162
2018-05-23    2162
2018-05-24    2162
2018-05-25    2162
2018-05-28    2162
dtype: int64

max date:
date
2025-12-29    2789
2025-12-24    2788
2025-12-26    2788
2025-12-30    2788
2026-01-02    2788
2026-01-05    2788
2026-01-06    2788
2025-12-23    2787
2025-12-22    2786
2026-01-07    2786
dtype: int64


In [84]:
# 05a2-49. date + ticker 중복 검사

duplicate_count = (
    stock_panel_clean[
        [
            "date",
            "ticker"
        ]
    ]
    .duplicated()
    .sum()
)

print(
    "duplicate date+ticker:",
    duplicate_count
)

duplicate date+ticker: 0


In [85]:
# 05a2-50. 전체 panel 결측치 검사

missing_summary = (
    stock_panel_clean
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

missing_summary

security_type    1970028
ticker                 0
date                   0
name                   0
market                 0
close                  0
change                 0
change_pct             0
open                   0
high                   0
low                    0
volume                 0
trading_value          0
market_cap             0
listed_shares          0
dtype: int64

In [86]:
# 05a2-51. 전체 기간 ohlc 논리 검증
# volume > 0 종목만 검사

traded_mask = (
    stock_panel_clean["volume"] > 0
)

invalid_ohlc_mask = (
    traded_mask
    &
    (
        (stock_panel_clean["high"] < stock_panel_clean["open"])
        |
        (stock_panel_clean["high"] < stock_panel_clean["close"])
        |
        (stock_panel_clean["low"] > stock_panel_clean["open"])
        |
        (stock_panel_clean["low"] > stock_panel_clean["close"])
    )
)

print(
    "invalid ohlc:",
    invalid_ohlc_mask.sum()
)


invalid ohlc: 11


In [87]:
stock_panel_clean.loc[
    invalid_ohlc_mask,
    [
        "date",
        "ticker",
        "name",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
].head(20)

,date,ticker,name,open,high,low,close,volume
536937,2019-02-11,056730,포스링크,0,0,0,1460,60795
701508,2019-05-29,310200,미래에셋대우스팩2호,0,0,0,2230,67503
702370,2019-05-29,270520,하나금융10호스팩,0,0,0,2085,58500
1073110,2020-01-28,033790,스카이문스테크놀로지,0,0,0,829,401
1476849,2020-10-07,349720,이베스트스팩5호,0,0,0,1995,147946
1663786,2021-02-01,141070,맥스로텍,0,0,0,1365,177
2081457,2021-10-14,336570,대신밸런스제8호스팩,0,0,0,2675,78
2280472,2022-02-09,215090,휴센텍,0,0,0,1505,1911
3035928,2023-04-24,089530,에이티세미콘,0,0,0,600,120
3631471,2024-03-28,001527,동양2우B,0,0,0,11480,2


In [89]:
# 05a2-52. 무거래 종목 데이터 확인

no_trade_mask = (
    stock_panel_clean["volume"] == 0
)

print(
    "no-trade rows:",
    no_trade_mask.sum()
)

print(
    "no-trade ratio:",
    f"{no_trade_mask.mean():.2%}"
)

no-trade rows: 182591
no-trade ratio: 3.46%


In [90]:
# 05a2-53. 시장 구분 확인

print(
    stock_panel_clean[
        "market"
    ]
    .value_counts(
        dropna=False
    )
)

market
KOSDAQ    3308550
KOSPI     1970028
Name: count, dtype: int64


In [91]:
market_by_date = (
    stock_panel_clean
    .groupby(
        [
            "date",
            "market"
        ]
    )
    .size()
    .unstack()
)

market_by_date.tail()

market,KOSDAQ,KOSPI
date,,
2026-09-09,1822,943
2026-09-10,1823,943
2026-09-11,1822,943
2026-09-14,1821,943
2026-09-15,1820,943


In [92]:
# 05a2-54. date + ticker 기준 정렬

stock_panel_clean = (
    stock_panel_clean
    .sort_values(
        [
            "date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)

In [95]:
# 05a2-55. parquet 사용 가능 여부 확인

try:

    import pyarrow

    print(
        "pyarrow available"
    )

except ImportError:

    print(
        "pyarrow not installed"
    )

pyarrow available


In [99]:
# 05a2-56. 전체 clean stock panel 저장
# pandas parquet 경로를 우회하고 pyarrow로 직접 저장

import pyarrow as pa
import pyarrow.parquet as pq


KRX_STOCK_PANEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "krx"
    / "krx_stock_panel_clean.parquet"
)


table = pa.Table.from_pandas(
    stock_panel_clean,
    preserve_index=False
)


pq.write_table(
    table,
    KRX_STOCK_PANEL_PATH,
    compression="snappy"
)


print(
    "saved:",
    KRX_STOCK_PANEL_PATH
)

saved: C:\code\portfolio_optimization\data\clean\krx\krx_stock_panel_clean.parquet


In [100]:
# 05a2-57. parquet 저장 결과 확인

panel_table = pq.read_table(
    KRX_STOCK_PANEL_PATH
)

panel_check = panel_table.to_pandas()


print(
    "shape:",
    panel_check.shape
)

print(
    "dates:",
    panel_check["date"].nunique()
)

print(
    "tickers:",
    panel_check["ticker"].nunique()
)

print(
    "duplicates:",
    panel_check[
        [
            "date",
            "ticker"
        ]
    ]
    .duplicated()
    .sum()
)

panel_check.head()

shape: (5278578, 15)
dates: 2112
tickers: 3186
duplicates: 0


,date,ticker,name,market,security_type,close,change,change_pct,open,high,low,volume,trading_value,market_cap,listed_shares
0,2018-02-05,000020,동화약품,KOSPI,None,10050,-250,-2.43,9900,10650,9900,313533,3216969180,280711273500,27931470
1,2018-02-05,000030,우리은행,KOSPI,None,16800,0,0.00,16700,17000,16550,1193269,20044289450,11356800000000,676000000
2,2018-02-05,000040,KR모터스,KOSPI,None,589,-20,-3.28,585,604,585,618818,366487410,70538443863,119759667
3,2018-02-05,000050,경방,KOSPI,None,14000,-450,-3.11,14500,14500,13900,36723,516341450,383813780000,27415270
4,2018-02-05,000060,메리츠화재,KOSPI,None,24700,-550,-2.18,24900,25200,24450,79722,1969614350,2725348600000,110338000


In [101]:
# 05a2-58. 데이터 품질 flag 추가

stock_panel_clean["is_no_trade"] = (
    stock_panel_clean["volume"] == 0
)

stock_panel_clean["is_invalid_ohlc"] = (
    (stock_panel_clean["volume"] > 0)
    &
    (
        (stock_panel_clean["high"] < stock_panel_clean["open"])
        |
        (stock_panel_clean["high"] < stock_panel_clean["close"])
        |
        (stock_panel_clean["low"] > stock_panel_clean["open"])
        |
        (stock_panel_clean["low"] > stock_panel_clean["close"])
    )
)

print(
    "no-trade:",
    stock_panel_clean["is_no_trade"].sum()
)

print(
    "invalid ohlc:",
    stock_panel_clean["is_invalid_ohlc"].sum()
)

no-trade: 182591
invalid ohlc: 11


In [102]:
# 05a2-59. 최종 clean panel 저장
# 원본 값은 유지하고 품질 flag만 추가

table = pa.Table.from_pandas(
    stock_panel_clean,
    preserve_index=False
)

pq.write_table(
    table,
    KRX_STOCK_PANEL_PATH,
    compression="snappy"
)

print(
    "saved:",
    KRX_STOCK_PANEL_PATH
)

saved: C:\code\portfolio_optimization\data\clean\krx\krx_stock_panel_clean.parquet


In [103]:
# 05a2-60. 최종 clean panel 검증

panel_table = pq.read_table(
    KRX_STOCK_PANEL_PATH
)

panel_check = panel_table.to_pandas()

print(
    "shape:",
    panel_check.shape
)

print(
    "dates:",
    panel_check["date"].nunique()
)

print(
    "tickers:",
    panel_check["ticker"].nunique()
)

print(
    "duplicates:",
    panel_check[
        ["date", "ticker"]
    ]
    .duplicated()
    .sum()
)

print(
    "no-trade:",
    panel_check["is_no_trade"].sum()
)

print(
    "invalid ohlc:",
    panel_check["is_invalid_ohlc"].sum()
)

shape: (5278578, 17)
dates: 2112
tickers: 3186
duplicates: 0
no-trade: 182591
invalid ohlc: 11
